In [1]:
from trendspy import Trends
import pandas as pd

tr = Trends()

termos_otimistas = ["abrir empresa", "comprar carro", "promoção passagens"]
termos_pessimistas = ["perder emprego", "inflação alta", "dívida"]

In [2]:
cesta_otimista = tr.interest_over_time(termos_otimistas, timeframe="2023-01-01 2026-08-06", geo="BR")
print(cesta_otimista.head())

            abrir empresa  comprar carro  promoção passagens  isPartial
time [UTC]                                                             
2023-01-01             19             43                  29      False
2023-01-08             21             40                  26      False
2023-01-15             22             40                  25      False
2023-01-22             21             43                  25      False
2023-01-29             22             40                  23      False


In [3]:
cesta_pessimista = tr.interest_over_time(termos_pessimistas, timeframe="2023-01-01 2026-08-06", geo="BR")
print(cesta_pessimista.head())

            perder emprego  inflação alta  dívida  isPartial
time [UTC]                                                  
2023-01-01               1              1      41      False
2023-01-08               1              0      38      False
2023-01-15               1              0      37      False
2023-01-22               1              1      38      False
2023-01-29               1              0      41      False


In [4]:
print(cesta_otimista.columns.tolist())
print(cesta_pessimista.columns.tolist())

['abrir empresa', 'comprar carro', 'promoção passagens', 'isPartial']
['perder emprego', 'inflação alta', 'dívida', 'isPartial']


In [5]:
cesta_otimista["media_otimista"] = cesta_otimista[["abrir empresa", "comprar carro", "promoção passagens"]].mean(axis=1)
cesta_pessimista["media_pessimista"] = cesta_pessimista[["perder emprego", "inflação alta", "dívida"]].mean(axis=1)

print(cesta_otimista[["media_otimista"]].head())
print(cesta_pessimista[["media_pessimista"]].head())

            media_otimista
time [UTC]                
2023-01-01       30.333333
2023-01-08       29.000000
2023-01-15       29.000000
2023-01-22       29.666667
2023-01-29       28.333333
            media_pessimista
time [UTC]                  
2023-01-01         14.333333
2023-01-08         13.000000
2023-01-15         12.666667
2023-01-22         13.333333
2023-01-29         14.000000


In [6]:
cesta_otimista["z_otimista"] = (cesta_otimista["media_otimista"] - cesta_otimista["media_otimista"].mean()) / cesta_otimista["media_otimista"].std()
cesta_pessimista["z_pessimista"] = (cesta_pessimista["media_pessimista"] - cesta_pessimista["media_pessimista"].mean()) / cesta_pessimista["media_pessimista"].std()

print(cesta_otimista[["z_otimista"]].head())
print(cesta_pessimista[["z_pessimista"]].head())

            z_otimista
time [UTC]            
2023-01-01    0.189536
2023-01-08   -0.120314
2023-01-15   -0.120314
2023-01-22    0.034611
2023-01-29   -0.275239
            z_pessimista
time [UTC]              
2023-01-01     -0.690334
2023-01-08     -0.988001
2023-01-15     -1.062418
2023-01-22     -0.913584
2023-01-29     -0.764751


In [7]:
ice_bruto = pd.DataFrame({
    "z_otimista": cesta_otimista["z_otimista"],
    "z_pessimista": cesta_pessimista["z_pessimista"]
})

print(ice_bruto.head())

            z_otimista  z_pessimista
time [UTC]                          
2023-01-01    0.189536     -0.690334
2023-01-08   -0.120314     -0.988001
2023-01-15   -0.120314     -1.062418
2023-01-22    0.034611     -0.913584
2023-01-29   -0.275239     -0.764751


In [8]:
from statsmodels.tsa.seasonal import STL

stl_otimista = STL(ice_bruto["z_otimista"], period=52, robust=True).fit()
stl_pessimista = STL(ice_bruto["z_pessimista"], period=52, robust=True).fit()

ice_bruto["z_otimista_dessaz"] = stl_otimista.trend + stl_otimista.resid
ice_bruto["z_pessimista_dessaz"] = stl_pessimista.trend + stl_pessimista.resid

print(ice_bruto.head())

            z_otimista  z_pessimista  z_otimista_dessaz  z_pessimista_dessaz
time [UTC]                                                                  
2023-01-01    0.189536     -0.690334          -0.656885            -1.150714
2023-01-08   -0.120314     -0.988001          -0.653318            -1.138566
2023-01-15   -0.120314     -1.062418          -0.649750            -1.126416
2023-01-22    0.034611     -0.913584          -0.646182            -1.114267
2023-01-29   -0.275239     -0.764751          -0.642614            -1.102116


In [11]:
stl_otimista = STL(ice_bruto["z_otimista"], period=52, robust=True).fit()
stl_pessimista = STL(ice_bruto["z_pessimista"], period=52, robust=True).fit()

ice_bruto["z_otimista_dessaz"] = stl_otimista.trend + stl_otimista.resid
ice_bruto["z_pessimista_dessaz"] = stl_pessimista.trend + stl_pessimista.resid

In [ ]:
ice_bruto["ICE"] = np.tanh(ice_bruto["z_otimista_dessaz"] - ice_bruto["z_pessimista_dessaz"])
print(ice_bruto["ICE"].describe())

['data', 'z_otimista', 'z_pessimista', 'z_otimista_dessaz', 'z_pessimista_dessaz', 'ICE']
        data       ICE
0 2023-01-01  0.457250
1 2023-01-08  0.450437
2 2023-01-15  0.443570
3 2023-01-22  0.436651
4 2023-01-29  0.429678


In [12]:
ice_final = ice_bruto.reset_index().rename(columns={"time [UTC]": "data"})
ice_final["data"] = pd.to_datetime(ice_final["data"]).astype("datetime64[ns]")
print(ice_final.columns.tolist())

['data', 'z_otimista', 'z_pessimista', 'z_otimista_dessaz', 'z_pessimista_dessaz', 'ICE']


In [13]:
eventos_ian = pd.read_csv("../data/eventos_com_ian.csv")
eventos_ian["data"] = pd.to_datetime(eventos_ian["data"]).astype("datetime64[ns]")

eventos_ian = eventos_ian.sort_values("data")
ice_final = ice_final.sort_values("data")

eventos_completo = pd.merge_asof(eventos_ian, ice_final[["data", "ICE"]], on="data", direction="backward")
print(eventos_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-03-14         0.000000  0.200000  0.387635
1  2023-04-12        -1.007506  0.311111  0.357169
2  2023-05-10         0.000000  0.177778  0.326825
3  2023-06-13        -1.007506  0.133333  0.287914
4  2023-07-12        -1.007506  0.200000  0.256062
5  2023-08-10         0.000000  0.200000  0.223625
6  2023-09-13         0.000000  0.122222  0.182342
7  2023-10-12         1.007506  0.088889  0.148786
8  2023-11-14        -1.007506  0.144444  0.106215
9  2023-12-12         1.007506  0.044444  0.071592
10 2024-01-11         1.007506  0.055556  0.535218
11 2024-02-13         1.007506  0.244444  0.767433
12 2024-03-12         0.000000  0.144444 -0.591468
13 2024-04-10         1.007506  0.311111 -0.096749
14 2024-05-15        -1.007506  0.244444 -0.260924
15 2024-06-12        -1.007506  0.111111 -0.328373
16 2024-07-11        -2.015012  0.177778 -0.091177
17 2024-08-14         0.000000  0.211111 -0.539211
18 2024-09-11         0.000000 

In [14]:
eventos_completo.to_csv("../data/eventos_completo.csv", index=False)